### use DistilBERT via Hugging Face to detect colors mentioned in a sentence, with view.

In [7]:
# download colors list dataset from kaggle
import kagglehub
import pandas
import pandas as pd

# Download the latest version
path = kagglehub.dataset_download("avi1023/color-names")

print("Path to dataset files:", path)

100%|██████████| 25.3k/25.3k [00:00<00:00, 1.44MB/s]

Extracting files...
Path to dataset files: C:\Users\wajee\.cache\kagglehub\datasets\avi1023\color-names\versions\1


In [11]:
!pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd

colors_list_path = r"../raw_data/color_names.csv"
colors_list = pd.read_csv(colors_list_path)
colors_list.head()

,Name,Hex (24 bit),Red (8 bit),Green (8 bit),Blue (8 bit),Hue (degrees),HSL.S (%),"HSL.L (%), HSV.S (%), HSV.V (%)"
0,Absolute zero,#0048BA,0,72,186,217.0,100.0,37.0
1,Acid green,#B0BF1A,176,191,26,65.0,76.0,43.0
2,Aero,#7CB9E8,124,185,232,206.0,70.0,70.0
3,Aero blue,#C9FFE5,201,255,229,151.0,100.0,89.0
4,African violet,#B284BE,178,132,190,288.0,31.0,63.0


In [7]:
colors_name_list = colors_list["Name"].to_list()

# for later coloring mapping
colors_list["Name"] = colors_list["Name"].str.strip().str.lower()
colors_list["Hex (24 bit)"] = colors_list["Hex (24 bit)"].str.strip()

# combine the Name alongside the Hex
color_hex_map = dict(zip(colors_list["Name"], colors_list["Hex (24 bit)"]))

In [8]:
print(colors_name_list[:10])
print(type(colors_name_list))

['Absolute zero', 'Acid green', 'Aero', 'Aero blue', 'African violet', 'Air Force blue (RAF)', 'Air Force blue (USAF)', 'Air superiority blue', 'Alabama crimson', 'Alabaster']
<class 'list'>


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [10]:
model_name = "valhalla/distilbart-mnli-12-3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: e1d74c49-d860-4567-b4f9-760fa2bef373)')' thrown while requesting HEAD https://huggingface.co/valhalla/distilbart-mnli-12-3/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [15]:
def detect_colors(text, candidate_colors, threshold=0.5):

    # Hypothesis template
    template = "This text is about {}."

    # Run zero-shot classification
    results = []
    for color in candidate_colors:
        hypothesis = template.format(color)
        inputs = tokenizer.encode_plus(text, hypothesis, return_tensors="pt", truncation=True)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=1)
            entailment_score = probs[0][2].item()  # index 2 = entailment
            results.append((color, entailment_score))

    # Filter and sort
    detected = [(c, s) for c, s in results if s > threshold]
    detected.sort(key=lambda x: x[1], reverse=True)
    return detected


In [16]:
# sample test 
sentence = "I have a red PC, it is wonderful."

colors_found = detect_colors(sentence, candidate_colors=colors_name_list)
for color, score in colors_found:
    print(f"{color}: {score:.3f}")

In [17]:
from IPython.display import display, HTML

def show_detected_colors(detected_colors):
    html = "<h4>Detected Colors:</h4><ul>"
    for color, score in detected_colors:
        hex_code = color_hex_map.get(color.lower(), "#ccc")  # fallback gray if not found
        html += f"""
        <li>
            <span style='display:inline-block;width:20px;height:20px;background-color:{hex_code};margin-right:10px;border:1px solid #000;'></span>
            <strong>{color}</strong>: {score}
        </li>
        """
    html += "</ul>"
    display(HTML(html))

In [18]:
sentence = "The sky was Aero blue and the flowers were Acid green."
colors_found = detect_colors(sentence, list(color_hex_map.keys()))
show_detected_colors(colors_found)